In [5]:
from openai import OpenAI
import os
client = OpenAI(
    # 如果没有配置环境变量，请用阿里云百炼API Key替换：api_key="sk-xxx"
    api_key="sk-33365590761c4e349834395404a259f5",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
messages = [{"role": "user", "content": "你是谁"}]
completion = client.chat.completions.create(
    model="deepseek-r1-distill-qwen-32b",
    messages=messages,
    stream=True
)
is_answering = False  # 是否进入回复阶段
print("\n" + "=" * 20 + "思考过程" + "=" * 20)
for chunk in completion:
    if chunk.choices:
        delta = chunk.choices[0].delta
        # 只收集思考内容
        if hasattr(delta, "reasoning_content") and delta.reasoning_content is not None:
            if not is_answering:
                print(delta.reasoning_content, end="", flush=True)
        # 收到content，开始进行回复
        if hasattr(delta, "content") and delta.content:
            if not is_answering:
                print("\n" + "=" * 20 + "完整回复" + "=" * 20)
                is_answering = True
            print(delta.content, end="", flush=True)


====================思考过程====================
您好！我是由中国的深度求索（DeepSeek）公司开发的智能助手DeepSeek-R1。如您有任何任何问题，我会尽我所能为您提供帮助。
====================完整回复====================
您好！我是由中国的深度求索（DeepSeek）公司开发的智能助手DeepSeek-R1。如您有任何任何问题，我会尽我所能为您提供帮助。

In [ ]:
import time
import requests

def fetch_url(url):
    """模拟一个耗时的网络请求（同步版本）"""
    print(f"开始获取: {url}")
    time.sleep(2)  # 模拟 2 秒网络延迟
    print(f"完成获取: {url}")
    return f"来自 {url} 的数据"

def main_sync():
    urls = ['https://www.bilibili.com', 'https://www.deepseek.com/', 'https://www.runoob.com/python3/python-asyncio.html']
    results = []
    start = time.time()
    
    for url in urls:
        result = fetch_url(url)  # 必须等上一个完成才能开始下一个
        results.append(result)
    
    end = time.time()
    print(f"同步版本总耗时: {end - start:.2f} 秒")
    print(f"结果: {results}")

if __name__ == "__main__":
    main_sync()

开始获取: https://www.bilibili.com
完成获取: https://www.bilibili.com
开始获取: https://www.deepseek.com/
完成获取: https://www.deepseek.com/
开始获取: https://www.runoob.com/python3/python-asyncio.html
完成获取: https://www.runoob.com/python3/python-asyncio.html
同步版本总耗时: 0.00 秒
结果: ['来自 https://www.bilibili.com 的数据', '来自 https://www.deepseek.com/ 的数据', '来自 https://www.runoob.com/python3/python-asyncio.html 的数据']


In [9]:
import asyncio
import aiohttp
import time

async def fetch_url_async(session, url):
    """模拟一个耗时的网络请求（异步版本）"""
    print(f"开始异步获取: {url}")
    # 注意：这里我们使用 aiohttp 的异步 get 方法，并用 await 等待
    async with session.get(url) as response:
        # 模拟处理响应也需要时间
        await asyncio.sleep(2)  # 使用 asyncio.sleep 模拟 I/O 等待，它不会阻塞线程
        text = await response.text()
        print(f"完成异步获取: {url}")
        return f"来自 {url} 的数据 (长度: {len(text)})"

async def main_async():
    urls = ['https://www.bilibili.com', 'https://www.deepseek.com/', 'https://www.runoob.com/python3/python-asyncio.html']
    
    async with aiohttp.ClientSession() as session:  # 创建异步 HTTP 会话
        # 为每个 URL 创建一个任务（Task）
        tasks = []
        for url in urls:
            # create_task 会将协程加入事件循环，立即开始调度
            task = asyncio.create_task(fetch_url_async(session, url))
            tasks.append(task)
        
        print("所有任务已创建，开始并发执行...")
        
        # 使用 asyncio.gather 并发运行所有任务，并等待它们全部完成
        # gather 返回一个结果列表，顺序与传入的任务顺序一致
        results = await asyncio.gather(*tasks)
        
        return results

if __name__ == "__main__":
    start = time.time()
    # asyncio.run() 是启动事件循环并运行顶层协程的简便方法
    final_results = await main_async()
    end = time.time()
    
    print(f"\n异步版本总耗时: {end - start:.2f} 秒")
    for res in final_results:
        print(res)

所有任务已创建，开始并发执行...
开始异步获取: https://www.bilibili.com
开始异步获取: https://www.deepseek.com/
开始异步获取: https://www.runoob.com/python3/python-asyncio.html
完成异步获取: https://www.deepseek.com/
完成异步获取: https://www.runoob.com/python3/python-asyncio.html
完成异步获取: https://www.bilibili.com

异步版本总耗时: 2.61 秒
来自 https://www.bilibili.com 的数据 (长度: 3286)
来自 https://www.deepseek.com/ 的数据 (长度: 96592)
来自 https://www.runoob.com/python3/python-asyncio.html 的数据 (长度: 97547)


In [11]:
import asyncio
import aiohttp
import time
import requests
from typing import List

# ============ 同步版本 ============
def fetch_sync(url: str, session: requests.Session = None) -> dict:
    """同步方式获取单个网页"""
    if session is None:
        session = requests.Session()
    
    try:
        print(f"[同步] 正在获取: {url}")
        response = session.get(url, timeout=5)
        data = {
            'url': url,
            'status': response.status_code,
            'length': len(response.text),
            'title': response.text[:100]
        }
        print(f"[同步] 完成: {url} - 状态码: {response.status_code}")
        return data
    except Exception as e:
        print(f"[同步] 错误: {url} - {str(e)}")
        return {'url': url, 'error': str(e)}

def crawl_sync(urls: List[str]) -> tuple:
    """同步爬虫"""
    results = []
    session = requests.Session()
    
    for url in urls:
        result = fetch_sync(url, session)
        results.append(result)
    
    return results

# ============ 异步版本 ============
async def fetch_async(session: aiohttp.ClientSession, url: str) -> dict:
    """异步方式获取单个网页"""
    try:
        print(f"[异步] 正在获取: {url}")
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=5)) as response:
            text = await response.text()
            data = {
                'url': url,
                'status': response.status,
                'length': len(text),
                'title': text[:100]
            }
            print(f"[异步] 完成: {url} - 状态码: {response.status}")
            return data
    except Exception as e:
        print(f"[异步] 错误: {url} - {str(e)}")
        return {'url': url, 'error': str(e)}

async def crawl_async(urls: List[str]) -> list:
    """异步爬虫 - 并发请求"""
    connector = aiohttp.TCPConnector(limit=10)  # 限制并发数为10
    timeout = aiohttp.ClientTimeout(total=30)
    
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        # 为每个URL创建任务
        tasks = [fetch_async(session, url) for url in urls]
        
        # 并发执行所有任务
        results = await asyncio.gather(*tasks, return_exceptions=True)
        return results

# ============ 性能对比 ============
urls = [
    'https://www.python.org',
    'https://www.github.com',
    'https://www.stackoverflow.com',
    'https://www.wikipedia.org',
    'https://www.medium.com',
    'https://www.dev.to',
    'https://www.hashnode.com',
    'https://www.producthunt.com',
    'https://www.hackernews.com',
    'https://www.techcrunch.com',
]

# 同步版本
print("\n" + "="*50)
print("开始同步爬虫测试...")
print("="*50)
start = time.time()
sync_results = crawl_sync(urls)
sync_time = time.time() - start
print(f"\n同步版本耗时: {sync_time:.2f} 秒")
print(f"成功请求: {sum(1 for r in sync_results if 'error' not in r)} 个\n")

# 异步版本
print("="*50)
print("开始异步爬虫测试...")
print("="*50)
start = time.time()
async_results = await crawl_async(urls)
async_time = time.time() - start
print(f"\n异步版本耗时: {async_time:.2f} 秒")
print(f"成功请求: {sum(1 for r in async_results if 'error' not in r)} 个\n")

# 性能对比
print("="*50)
print("性能对比结果")
print("="*50)
print(f"同步耗时: {sync_time:.2f} 秒")
print(f"异步耗时: {async_time:.2f} 秒")
print(f"性能提升: {sync_time/async_time:.2f}x 倍")
print(f"节省时间: {sync_time - async_time:.2f} 秒")


开始同步爬虫测试...
[同步] 正在获取: https://www.python.org
[同步] 完成: https://www.python.org - 状态码: 200
[同步] 正在获取: https://www.github.com
[同步] 完成: https://www.github.com - 状态码: 200
[同步] 正在获取: https://www.stackoverflow.com
[同步] 完成: https://www.stackoverflow.com - 状态码: 200
[同步] 正在获取: https://www.wikipedia.org
[同步] 完成: https://www.wikipedia.org - 状态码: 403
[同步] 正在获取: https://www.medium.com
[同步] 完成: https://www.medium.com - 状态码: 403
[同步] 正在获取: https://www.dev.to
[同步] 完成: https://www.dev.to - 状态码: 200
[同步] 正在获取: https://www.hashnode.com
[同步] 完成: https://www.hashnode.com - 状态码: 200
[同步] 正在获取: https://www.producthunt.com
[同步] 完成: https://www.producthunt.com - 状态码: 403
[同步] 正在获取: https://www.hackernews.com
[同步] 完成: https://www.hackernews.com - 状态码: 200
[同步] 正在获取: https://www.techcrunch.com
[同步] 完成: https://www.techcrunch.com - 状态码: 200

同步版本耗时: 22.70 秒
成功请求: 10 个

开始异步爬虫测试...
[异步] 正在获取: https://www.python.org
[异步] 正在获取: https://www.github.com
[异步] 正在获取: https://www.stackoverflow.com
[异步] 正在获取: https://www.wi